In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "16"
os.environ["OPENBLAS_NUM_THREADS"] = "16"
os.environ["MKL_NUM_THREADS"] = "16"

In [11]:
import pandas as pd 
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

In [47]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix, recall_score, accuracy_score, precision_score

In [ ]:
df = pd.read_csv("~/mission_sih/mission_sih/data/type_0_dataset.csv")

category
Forest         2527885
Agriculture     898942
Name: count, dtype: int64

In [14]:

df["category"].value_counts(dropna=False)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3426827 entries, 0 to 3426826
Data columns (total 14 columns):
 #   Column        Dtype  
---  ------        -----  
 0   latitude      float64
 1   longitude     float64
 2   brightness    float64
 3   scan          float64
 4   track         float64
 5   acq_time      int64  
 6   confidence    int64  
 7   bright_t31    float64
 8   frp           float64
 9   daynight      int64  
 10  type          int64  
 11  year          int64  
 12  match_dist_m  float64
 13  category      str    
dtypes: float64(8), int64(5), str(1)
memory usage: 366.0 MB


In [13]:
df.drop(columns="acq_date", inplace=True)

In [20]:
df = df.replace({"category":{"Forest":1, "Agriculture":2}})
df.head()

,latitude,longitude,brightness,scan,track,acq_time,confidence,bright_t31,frp,daynight,type,year,match_dist_m,category
0,23.04102,92.52061,336.69,0.62,0.71,606,1,296.72,6.39,0,0,2018,307.030507,1
1,22.50259,92.55136,330.26,0.63,0.72,606,1,299.59,8.80,0,0,2018,313.398197,1
2,22.50408,92.55746,346.15,0.63,0.72,606,1,298.87,10.10,0,0,2018,548.842875,1
3,22.41149,92.65405,330.42,0.63,0.72,606,0,300.69,3.30,0,0,2018,102.962301,1
4,22.12835,92.67033,330.93,0.63,0.72,606,1,297.33,30.87,0,0,2018,421.911597,1


In [21]:
scaler = StandardScaler()

df_scaled = scaler.fit_transform(df)

In [30]:
df["category"] = df['category'].astype(int)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3426827 entries, 0 to 3426826
Data columns (total 14 columns):
 #   Column        Dtype  
---  ------        -----  
 0   latitude      float64
 1   longitude     float64
 2   brightness    float64
 3   scan          float64
 4   track         float64
 5   acq_time      int64  
 6   confidence    int64  
 7   bright_t31    float64
 8   frp           float64
 9   daynight      int64  
 10  type          int64  
 11  year          int64  
 12  match_dist_m  float64
 13  category      int64  
dtypes: float64(8), int64(6)
memory usage: 366.0 MB


In [31]:
X = df.drop(columns="category")
y = df["category"]

In [32]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [29]:
y_train.head(10)

2141179    2
2896208    2
729453     2
2023458    1
217026     1
2239663    1
1143078    1
3236183    2
1626688    1
2493720    1
Name: category, dtype: object

In [33]:
clf = MLPClassifier(
hidden_layer_sizes=(200,),
activation="relu",
solver='adam',
alpha=0.0001,
batch_size=1000, #############
learning_rate='adaptive',
learning_rate_init=0.001, #############
power_t=0.5,
max_iter=1000, #################
shuffle=True,
random_state=42,
tol=0.0001,
verbose=True,
warm_start=True, #############
momentum=0.9,
nesterovs_momentum=True,
early_stopping=True,
validation_fraction=0.1,
beta_1=0.9,
beta_2=0.999,
epsilon=1e-08,
n_iter_no_change=10,
max_fun=15000
)

In [35]:
model = clf.fit(X_train, y_train)

Iteration 1, loss = 0.85535020
Validation score: 0.739286
Validation score did not improve more than tol=0.000100 for 10 consecutive epochs. Stopping.


In [36]:
y_pred = model.predict(X_test)

In [40]:
confusion_matrix(y_test, y_pred)

array([[495286,  10323],
       [168216,  11541]])

In [52]:
f1_score(y_test, y_pred)

0.8472865279686873

In [43]:
recall_score(y_test, y_pred)

0.9795830374854878

In [48]:
precision_score(y_test, y_pred)

0.7464725049811455

In [45]:
accuracy_score(y_test, y_pred)

0.7394983118508943

In [46]:
model.score(X_test, y_test)

0.7394983118508943

In [53]:
import joblib
joblib.dump(model, "hf_model/type_0_model.joblib")

['hf_model/type_0_model.joblib']

In [54]:
import os
from dotenv import load_dotenv
load_dotenv()


True

In [55]:
hf_token = os.getenv("HUGGING_FACE_API")

In [56]:
from huggingface_hub import HfApi, create_repo

# Your Hugging Face repo
REPO_ID = "bazik-0/mission-sih-type-0"


with open("hf_model/model_predict.py", "w") as f:
    f.write("""
import joblib
import pandas as pd

model = joblib.load("type_0_model.joblib")

FEATURES = [
    "latitude",
    "longitude",
    "scan",
    "track",
    "track_scan",
    "frp",
    "radiation",
    "brightness",
    "bright_t31",
    "final_bright",
    "acq_time",
    "daynight",
    "confidence",
    "type",
    "match_dist_m"
]

def predict(data):
    X = pd.DataFrame([data])[FEATURES]
    return model.predict(X)[0]

def predict_proba(data):
    X = pd.DataFrame([data])[FEATURES]
    return model.predict_proba(X)[0].tolist()
""")

# 3. Requirements
with open("hf_model/requirements.txt", "w") as f:
    f.write("""scikit-learn
pandas
numpy
joblib
""")
api = HfApi(token=hf_token)

# 4. Create HF repo
api.create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    exist_ok=True
)

# 5. Upload everything

api.upload_folder(
    folder_path="hf_model",
    repo_id=REPO_ID,
    repo_type="model"
)

print("Uploaded successfully!")
print(f"https://huggingface.co/{REPO_ID}")

Uploaded successfully!
https://huggingface.co/bazik-0/mission-sih-type-0
